In [31]:
from elasticsearch import Elasticsearch, helpers
import json 
bulk = helpers.bulk # Insertion de masse 

es = Elasticsearch("http://elasticsearch:9200")  # connexion au server es depuis Jupyter 

In [32]:
# Suite du cours 
query = {
  "size": 0,
  "aggs": {
    "par_piece": {
      "terms": {
        "field": "play_name.keyword"
      }
    }
  }
}

res = es.search(index="shakespeare", body=query)

res["aggregations"]

{'par_piece': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'Hamlet', 'doc_count': 8},
   {'key': 'Macbeth', 'doc_count': 7},
   {'key': 'Othello', 'doc_count': 6},
   {'key': 'Romeo and Juliet', 'doc_count': 3},
   {'key': 'King Lear', 'doc_count': 2},
   {'key': 'Julius Caesar', 'doc_count': 1}]}}

### Compter le nombre de documents par speaker

In [65]:
# Pas de fonction count avec Elastic 
query = {
  "aggs": {
    "per_speaker": {
      "terms": {
        "field": "speaker.keyword"
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)

res["aggregations"]

{'per_speaker': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'HAMLET', 'doc_count': 8},
   {'key': 'MACBETH', 'doc_count': 4},
   {'key': 'OTHELLO', 'doc_count': 4},
   {'key': 'LADY MACBETH', 'doc_count': 3},
   {'key': 'ROMEO', 'doc_count': 3},
   {'key': 'IAGO', 'doc_count': 2},
   {'key': 'KING LEAR', 'doc_count': 2},
   {'key': 'MARK ANTONY', 'doc_count': 1}]}}

### Calculer l'année moyenne

In [34]:
query = {
   "aggs": {
        "avg_year": {
            "avg": {
                "field": "year"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

round(res["aggregations"]["avg_year"]["value"])

1602

### Trouver l'année min et max

In [66]:
query = {
   "aggs": {
        "min_year": {
            "min": {
                "field": "year"
            }
        },
        "max_year": {
            "max": {
                "field": "year"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

print(int( res["aggregations"]["min_year"]["value"] ) )
print(int( res["aggregations"]["max_year"]["value"] ) )

1595
1606


### Compter le nombre de documents par pièce entre 1600 et 1610

In [67]:
query = {
    "query": {
        "range": {
            "year": {
                "gte": 1600,
                "lte": 1610
            }
        }
    },
    "aggs": {
        "per_play": {
            "terms": {
                "field": "play_name.keyword"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

print( json.dumps( res["aggregations"], indent=2 ) )

{
  "per_play": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "Hamlet",
        "doc_count": 4
      },
      {
        "key": "Macbeth",
        "doc_count": 4
      },
      {
        "key": "Othello",
        "doc_count": 4
      },
      {
        "key": "King Lear",
        "doc_count": 2
      }
    ]
  }
}


### Calculer l'année moyenne par speaker

In [76]:
query = {
    "aggs": {
        "per_speaker": {
              "terms": {
                "field": "speaker.keyword"
        },
          "aggs": {
            "avg_year": {
                  "avg": {
                    "field": "year"
                  }
            }
          }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

# print(res["aggregations"]["per_speaker"]["buckets"])

g = ( (d["key"], int(d["avg_year"]["value"]) ) for d in res["aggregations"]["per_speaker"]["buckets"] )


for v in g:
    print(v)




('HAMLET', 1602)
('MACBETH', 1606)
('OTHELLO', 1604)
('LADY MACBETH', 1606)
('ROMEO', 1595)
('IAGO', 1604)
('KING LEAR', 1605)
('MARK ANTONY', 1599)


In [61]:
g = ( (d["key"], d["doc_count"], int( d["moyenne_annee"]["value"] ) ) for d in res["aggregations"]["per_speaker"]["buckets"] )

In [62]:
for r in g:
    print(r)

('HAMLET', 8, 1602)
('MACBETH', 4, 1606)
('OTHELLO', 4, 1604)
('LADY MACBETH', 3, 1606)
('ROMEO', 3, 1595)
('IAGO', 2, 1604)
('KING LEAR', 2, 1605)
('MARK ANTONY', 1, 1599)


### Regrouper les documents par période (avant 1600, 1600–1605, après 1605)

In [64]:
query = {
    "aggs": {
        "per_range": {
            "range": {
                "field": "year",
                "ranges": [
                    {"to": 1600},                 # avant 1600
                    {"from": 1600, "to": 1605},   # 1600–1605
                    {"from": 1605}                # après 1605
                ]
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

print( json.dumps( res["aggregations"], indent=2 ) )

{
  "per_range": {
    "buckets": [
      {
        "key": "*-1600.0",
        "to": 1600.0,
        "doc_count": 4
      },
      {
        "key": "1600.0-1605.0",
        "from": 1600.0,
        "to": 1605.0,
        "doc_count": 8
      },
      {
        "key": "1605.0-*",
        "from": 1605.0,
        "doc_count": 6
      }
    ]
  }
}
